In [53]:
import requests
from tqdm import tqdm
import pandas as pd
import psycopg2

from datetime import datetime, timedelta

from joblib import load, dump

from evidently import Dataset, DataDefinition, Report, Regression
from evidently.metrics import ValueDrift, DatasetMissingValueCount, UniqueValueCount, QuantileValue
from evidently.presets import DataDriftPreset
from evidently.generators import ColumnMetricGenerator

from evidently.tests import *

In [5]:
files = [("green_tripdata_2024-03.parquet", "./data")]

print("Download files:")
for file, path in files:
    url=f"https://d37ci6vzurychx.cloudfront.net/trip-data/{file}"
    resp=requests.get(url, stream=True)
    save_path=f"{path}/{file}"
    with open(save_path, "wb") as handle:
        for data in tqdm(
                resp.iter_content(),
                desc=f"{file}",
                postfix=f"save to {save_path}",
                total=int(resp.headers["Content-length"])
        ):
            handle.write(data)
           

Download files:


green_tripdata_2024-03.parquet: 100%|██████████| 1372372/1372372 [00:10<00:00, 133479.21it/s, save to ./data/green_tripdata_2024-03.parquet]


### Prepare dataset

In [7]:
prod_df = pd.read_parquet('data/green_tripdata_2024-03.parquet')

In [8]:
prod_df.shape[0]

57457

### Add Metric

In [57]:
ref_data = pd.read_parquet('data/reference.parquet')

In [12]:
with open('models/lin_reg.bin', 'rb') as f_in:
    model = load(f_in)

In [13]:
target = "duration_min"
num_features = ["passenger_count", "trip_distance", "fare_amount", "total_amount"]
cat_features = ["PULocationID", "DOLocationID"]

In [21]:
prod_df['prediction'] = model.predict(prod_df[num_features + cat_features].fillna(0))

In [16]:
data_definition = DataDefinition(
    numerical_columns=num_features + ["prediction"],
    categorical_columns=cat_features,
    regression=[Regression(target=None, prediction="prediction")],
    
    )

In [20]:
report = Report(
    metrics=[
        ColumnMetricGenerator(ValueDrift, 
                          columns=["prediction"],
                          metric_kwargs={"method":"psi"}),
        ColumnMetricGenerator(UniqueValueCount, 
                          column_types='cat'),
        ColumnMetricGenerator(
            QuantileValue,
            columns=["fare_amount"], metric_kwargs={"quantile":0.5} ),
        DataDriftPreset(),
        DatasetMissingValueCount()
    ], include_tests=True
)

In [22]:
reference_data = Dataset.from_pandas(ref_data, data_definition=data_definition)
current_data = Dataset.from_pandas(prod_df, data_definition=data_definition)

In [31]:
march_taxi_report = report.run(reference_data=reference_data, current_data=current_data)

In [26]:
begin = datetime(2024, 3, 1, 0, 0)

In [41]:
import random

SEND_TIMEOUT = 10
rand = random.Random()

create_table_statement = """
    drop table if exists dummy_metrics;
    create table dummy_metrics(
        timestamp timestamp,
        prediction_drift float,
        num_drifted_columns integer,
        share_missing_value float,
        fare_amount_quantile_value float
    );
"""

In [42]:
conn = psycopg2.connect("host=localhost port=5432 user=postgres password=example")
conn.autocommit = True  # Required for CREATE DATABASE
try:
    with conn.cursor() as cur:
        cur.execute("SELECT 1 FROM pg_database WHERE datname='test'")
        if len(cur.fetchall()) == 0:
            cur.execute("CREATE DATABASE test")
finally:
    conn.close()

# Second connection to the test database for table creation
with psycopg2.connect("host=localhost port=5432 dbname=test user=postgres password=example") as conn:
    with conn.cursor() as cur:
        cur.execute(create_table_statement)

In [43]:
march_taxi_report_dict = march_taxi_report.dict()

In [62]:
def calculate_metrics_psql(curr, i):
    current_data = prod_df[(prod_df.lpep_pickup_datetime <= (begin + timedelta(i))) & (prod_df.lpep_pickup_datetime < (begin + timedelta(i + 1)))]

    current_data = current_data.fillna(0).copy()
    current_data['prediction'] = model.predict(current_data[num_features + cat_features])

    report_reference_data = Dataset.from_pandas(ref_data, data_definition=data_definition)
    report_current_data = Dataset.from_pandas(current_data, data_definition=data_definition)

    taxi_report = report.run(reference_data= report_reference_data, current_data = report_current_data)

    taxi_report_dict = taxi_report.dict()
    

    prediction_drift = taxi_report_dict['metrics'][0]['value']
    num_drifted_columns = taxi_report_dict['metrics'][4]['value']['count']
    share_of_missing_vals = taxi_report_dict['metrics'][-1]['value']['share']
    fare_amount_quantile_val = taxi_report_dict['metrics'][3]['value']
    
    print('inserting record')
    curr.execute(
        "insert into dummy_metrics(timestamp ,prediction_drift, num_drifted_columns, share_missing_value, fare_amount_quantile_value) values (%s, %s, %s, %s, %s)",
        (begin + timedelta(i), prediction_drift, num_drifted_columns, share_of_missing_vals, fare_amount_quantile_val )
    )

In [65]:
import logging
import time 

last_send = datetime.now() - timedelta(seconds=10)

with psycopg2.connect("host=localhost port=5432 dbname=test user=postgres password=example") as conn:
    for i in range(0, 32):
        with conn.cursor() as curr:
            calculate_metrics_psql(curr, i)

        new_send = datetime.now()
        seconds_elapsed = (new_send - last_send).total_seconds()
        if seconds_elapsed < SEND_TIMEOUT:
            time.sleep(SEND_TIMEOUT - seconds_elapsed)
        while last_send < new_send:
            last_send = last_send + timedelta(seconds=10)
        logging.info("data sent")


/workspaces/mlops/venv/lib/python3.12/site-packages/evidently/legacy/calculations/data_drift.py:462: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/workspaces/mlops/venv/lib/python3.12/site-packages/evidently/legacy/calculations/data_drift.py:462: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



inserting record


/workspaces/mlops/venv/lib/python3.12/site-packages/evidently/legacy/calculations/data_drift.py:462: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/workspaces/mlops/venv/lib/python3.12/site-packages/evidently/legacy/calculations/data_drift.py:462: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



inserting record


/workspaces/mlops/venv/lib/python3.12/site-packages/evidently/legacy/calculations/data_drift.py:462: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/workspaces/mlops/venv/lib/python3.12/site-packages/evidently/legacy/calculations/data_drift.py:462: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



...

inserting record


/workspaces/mlops/venv/lib/python3.12/site-packages/evidently/legacy/calculations/data_drift.py:462: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/workspaces/mlops/venv/lib/python3.12/site-packages/evidently/legacy/calculations/data_drift.py:462: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



inserting record


/workspaces/mlops/venv/lib/python3.12/site-packages/evidently/legacy/calculations/data_drift.py:462: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

/workspaces/mlops/venv/lib/python3.12/site-packages/evidently/legacy/calculations/data_drift.py:462: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



inserting record
